# Module 1: RAG Concepts

**Duration:** 50 minutes

In this module we look at what RAG is, why it exists, and how the pieces fit together.
Then we build a minimal version from scratch using only numpy, so every step is visible
before we start using libraries to handle it for us.

**By the end of this module you can:**
- explain in your own words what problem RAG solves and when it is (not) the right tool
- name the steps of a RAG pipeline: encode, retrieve, build the prompt, generate
- run and modify a minimal RAG system written from scratch

---

## 1.0 Warm-up: ask the model directly

Before any theory, let's ask a local LLM a question about a real organisation.
Run the cell and read the answer carefully. Is it right? How would you know?
Keep the answer in mind, we come back to it at the end of section 1.4.

If your installation is still downloading the model, watch the trainer's screen and run the
cell later; you will need it again in section 1.4.

In [ ]:
import json
from os import getenv
from urllib.parse import urljoin

import requests

OLLAMA_URL = urljoin(getenv('OLLAMA_HOST', 'http://localhost:11434'), 'api')
MODEL = 'llama3.2:3b'


def generate(prompt: str, temp: float = 0.3) -> str:
    url = OLLAMA_URL + '/generate'
    data = {'model': MODEL, 'prompt': prompt, 'stream': False,
            'options': {'temperature': temp}}
    r = requests.post(url, json=data)
    return json.loads(r.text).get('response', '')


WARMUP_QUESTION = (
    'When do the AI Service Center Berlin-Brandenburg paper reading sessions take place?'
)
print(generate(WARMUP_QUESTION))

## 1.1 Why RAG?

Large language models are trained on a fixed snapshot of text. Once training is done,
their knowledge is frozen. This creates a few practical problems:

- They have no access to your private documents.
- They cannot answer questions about events after their training cutoff.
- Retraining a model to update its knowledge is expensive and slow.

Retrieval Augmented Generation (RAG) sidesteps all of this. Instead of trying to bake
knowledge into the model weights, we retrieve the relevant information at query time
and hand it to the model as part of the prompt. The model does not need to remember
anything, it just needs to read and reason.

### Two kinds of memory in LLMs

It helps to think of an LLM as having two distinct memory systems:

| Memory type | Where it lives | How it's updated | Example |
|-------------|----------------|------------------|---------|
| **Parametric memory** | Model weights | Re-training or fine-tuning (expensive) | "Paris is the capital of France" |
| **Non-parametric memory** | External store (retrieved at runtime) | Add/update documents in the vector DB | Your internal wiki, today's news |

RAG is a way to give a model access to non-parametric memory without touching
the weights. The model becomes an excellent *reader* and *reasoner*, while you are the one that supplies
the *facts*.

### When to use RAG (and when not to)

**Good fit for RAG:**
- Querying internal company documents, wikis, or support tickets
- Answering questions about content that changes frequently (pricing, policies, news)
- Personalised responses grounded in a specific user's data
- Reducing hallucinations by giving the model a source to read from
- Narrow domains where an off-the-shelf model has no training data

**RAG is not the right tool when:**
- The task is purely generative (creative writing, summarisation of provided text)
- You need the model to perform reasoning that doesn't depend on external facts
- Your documents are very short and could just go directly into the prompt context
- Fine-tuning is more appropriate (e.g. learning a new writing style or output format)

> **Rule of thumb:** if you find yourself wishing the model *knew* something it doesn't,
> RAG can usually help. If you're wishing the model *behaved* differently, fine-tuning
> is usually the answer.

### RAG vs fine-tuning vs prompt stuffing

Three common approaches to knowledge injection, compared:

| Approach | Latency | Cost | Staleness | Private data |
|----------|---------|------|-----------|--------------|
| Prompt stuffing (paste docs into context) | Low | Medium (long contexts) | Never stale | ✓ |
| Fine-tuning | Low (no retrieval) | Very high | Stale until retrained | Risk of memorisation |
| RAG | Medium (retrieval step) | Low (add/remove docs) | Always fresh | ✓ |

RAG is usually the best starting point for document Q&A because it is cheap to
update, keeps your data in one place, and lets you inspect exactly what the model
was given.


## 1.2 How it works

The full pipeline has two phases.

**Indexing phase**:

<img src="../images/indexing_phase.svg" width="600" />

**Query phase**:

<img src="../images/query_phase.svg" width="600" />

The key insight is that the retrieval step happens in **vector space**.
Text gets turned into numbers, and similarity between texts becomes
a geometric distance between points. We will look closely at how
this works in Module 2.

### The context window as working memory

Think of the LLM's context window as a piece of paper on your desk.
The model can only read what is written on that paper right now.
RAG is the process of *choosing what to write on the paper* before
handing it to the model.

- If you write the right thing → the model answers correctly.
- If you write too much → the model loses focus (the "lost in the middle" problem).
- If you write nothing → the model falls back on parametric memory (and may hallucinate).

This framing makes many RAG design decisions obvious:
chunking controls the granularity of what you can write;
retrieval controls which pieces you select;
prompt construction controls how you organise the page.

### Where does the LLM actually run?

In this workshop we use **Ollama**, which runs a quantised LLM locally on your machine.
The model weights are stored on disk and loaded into RAM (or VRAM if you have a GPU).
Ollama exposes an HTTP API on `localhost:11434` and our Python code calls that API.

This is why no API key is needed: everything runs on your own hardware.
The trade-off is that the model is much smaller than commercial APIs (3 billion
parameters here, versus hundreds of billions for large commercial models), but for most RAG
use cases that is fine because the model's main job is *reading and summarising*, not
*reasoning from scratch*.

## 1.3 The anatomy of a RAG prompt

Before we build the code, let's look at what a RAG prompt actually looks like.
Most RAG prompts follow a simple template:

```
[Instruction]
Use only the following context to answer the question.
If the answer is not in the context, say "I don't know".

[Retrieved context]
Context:
The AI Service Center paper reading sessions take place online every Wednesday afternoon.
The AI Service Center Berlin-Brandenburg offers workshops, consulting, and compute resources.

[User question]
Question: When are the paper reading sessions?
Answer:
```

This is the prompt our `rag()` function in section 1.4 builds, with the two best chunks from
our tiny corpus.

The quality of this prompt depends on:
1. **Which chunks were retrieved** (retrieval quality)
2. **How many chunks** (too few = missing info; too many = context overload)
3. **The instruction phrasing** (affects whether the model stays grounded)

The instruction "answer using only the context" is crucial. Without it, smaller
models often ignore the provided context and hallucinate from their training data.

In Module 4 we switch to the `RAGTool.get_context_prompt()` method, which builds the same kind
of prompt with slightly different wording.

---

## 1.4 RAG from scratch

Before using any libraries, we build a minimal RAG system using only numpy.
It will not scale, but it makes every step explicit.

We need:
1. A way to encode text as vectors.
2. A way to find the most similar vector to a query.
3. A way to call an LLM with the retrieved context.

In [ ]:
import warnings

from tqdm import TqdmExperimentalWarning

warnings.filterwarnings('ignore', category=TqdmExperimentalWarning)

import numpy as np
from sentence_transformers import SentenceTransformer

EMBEDDING_MODEL = 'all-MiniLM-L6-v2'
model = SentenceTransformer(EMBEDDING_MODEL)
print('Model loaded.')


In [ ]:
# Our tiny document corpus
documents = [
    "Bali's beautiful beaches and rich culture make it a popular travel destination.",
    "Pizza in Rome is famous for its thin crust, fresh ingredients, and wood-fired ovens.",
    "Graphics processing units have become essential for training AI models.",
    "Newton's laws of motion transformed our understanding of physics.",
    "The French Revolution played a crucial role in shaping contemporary France.",
    "Maintaining good health requires regular exercise, a balanced diet, and quality sleep.",
    "Global warming threatens ecosystems and wildlife across the planet.",
    "The AI Service Center Berlin-Brandenburg offers workshops, consulting, "
    "and compute resources.",
    "Django Reinhardt's jazz compositions are celebrated for their captivating melodies.",
    "The AI Service Center paper reading sessions take place online every Wednesday afternoon.",
]

# Encode all documents into vectors
doc_embeddings = model.encode(documents)

print('Shape of embedding matrix:', doc_embeddings.shape)
print('Each document becomes a vector of', doc_embeddings.shape[1], 'numbers.')

Each document is now a point in a 384-dimensional space.
Documents that talk about similar things should be nearby in this space.
That is the core idea behind semantic search.


In [ ]:
# Cosine similarity measures the angle between two vectors.
# Two vectors pointing in the same direction have similarity 1.
# Two perpendicular vectors have similarity 0.

def cosine_similarity(a: np.ndarray, b: np.ndarray) -> float:
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))


def retrieve(query: str, top_k: int = 3) -> list[tuple[str, float]]:
    query_embedding = model.encode(query)
    scores = [cosine_similarity(query_embedding, doc_emb) for doc_emb in doc_embeddings]
    ranked = sorted(zip(documents, scores), key=lambda x: x[1], reverse=True)
    return ranked[:top_k]


query = 'I want to learn about artificial intelligence in Berlin'
results = retrieve(query)

print(f'Query: {query}\n')
for doc, score in results:
    print(f'  score {score:.3f}: {doc}')


In [ ]:
# Now add the LLM step (generate() was defined in the warm-up cell)

def rag(query: str, top_k: int = 3) -> str:
    # Step 1: retrieve
    results = retrieve(query, top_k=top_k)
    context = '\n'.join(doc for doc, _ in results)

    # Step 2: build prompt
    prompt = (
        'Use only the following context to answer the question. '
        'If the answer is not in the context, say "I don\'t know". '
        'After your answer, quote the sentence you used, starting with "Reference:".\n\n'
        f'Context:\n{context}\n\n'
        f'Question: {query}\n'
        'Answer:'
    )

    # Step 3: generate
    return generate(prompt)


print(rag('Where can you eat good pizza?'))
print()
print('Warm-up question, now with retrieval:')
print(rag(WARMUP_QUESTION))

That is the full pipeline: encode, retrieve, prompt, generate.
Everything in the rest of the workshop is an improvement on one of those four steps.

Compare the warm-up answer with the answer you got with retrieval. The model did not get
smarter: it was given a sentence to read.

---

**Exercises**

1. Change the query to something unrelated to the documents (e.g. 'Who invented the telephone?').
   What does the retriever return? What does the LLM say?

2. Add three new documents to the `documents` list on a topic of your choice,
   re-encode, and ask a question about them.

3. Print the full prompt that gets sent to the LLM. Does reading it help explain the answer?

4. If you have pulled other models with Ollama (`ollama list`), change `MODEL` in the warm-up
   cell. Do the answers change significantly with the model choice?

5. **Explain it to your neighbour (3 minutes):** in two or three sentences, explain what RAG
   does, using the "piece of paper on the desk" picture from section 1.2. Then swap roles.

---

**Further reading**

- Original RAG paper: https://arxiv.org/abs/2005.11401
- Sentence Transformers: https://www.sbert.net/